In [1]:
%pip install pymupdf

  Using cached pymupdf-2.0-cp311-abi3-win_amd64.whl.metadata (25 kB)
  Using cached pymupdf_layout-2.0-py3-none-any.whl.metadata (368 bytes)
  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
  Using cached pymupdf4llm-2.0-py3-none-any.whl.metadata (360 bytes)
Using cached pymupdf-2.0-cp311-abi3-win_amd64.whl (63.0 MB)
Using cached pymupdf_layout-2.0-py3-none-any.whl (927 bytes)
Using cached pymupdf4llm-2.0-py3-none-any.whl (904 bytes)
   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.9 MB 


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys, json
sys.path.append("..")
from pathlib import Path
from collections import Counter
import numpy as np
from sentence_transformers import SentenceTransformer
from src.ingest import build_chunks

DOCS_DIR = Path("../rag/docs")
OUT_DIR = Path("../rag/index_public")          # public index ithe save hoil
OUT_DIR.mkdir(parents=True, exist_ok=True)

# fakt hya 5 files public index madhe jaatil. NKF nahi (parvangi nahi)
PUBLIC_SOURCES = {
    "CDC Basics.pdf", "CDC Risk Factors.pdf", "CDC Testing.pdf", "CDC Living with CKD.pdf",
    "KDIGO 2024 Executive Summary.pdf",
}

In [3]:
chunks = build_chunks(DOCS_DIR, PUBLIC_SOURCES)
lens = [len(c["text"]) for c in chunks]

print("TOTAL chunks:", len(chunks))
print(Counter(c["source"] for c in chunks))
print("min/avg/max length:", min(lens), sum(lens) // len(lens), max(lens))
print("start with lowercase:", sum(c["text"][0].islower() for c in chunks))
assert not any("NKF" in c["source"] for c in chunks), "NKF file public index madhe aali!"

# faltu text rahila ka (sagle 0 pahije)
for kw in ["(Continued", "\ufb01", "modi fi", "| CDC", "consultancy fees"]:
    print(repr(kw), "->", sum(kw in c["text"] for c in chunks))
print(chunks[0])

TOTAL chunks: 154
Counter({'KDIGO 2024 Executive Summary.pdf': 131, 'CDC Basics.pdf': 8, 'CDC Testing.pdf': 7, 'CDC Living with CKD.pdf': 5, 'CDC Risk Factors.pdf': 3})
min/avg/max length: 144 743 900
start with lowercase: 11
'(Continued' -> 0
'ﬁ' -> 0
'modi fi' -> 0
'| CDC' -> 0
'consultancy fees' -> 0
{'source': 'CDC Basics.pdf', 'page': 1, 'text': "Chronic Kidney Disease Basics KEY POINTS More than 1 in 7 American adults has chronic kidney disease (CKD), and as many as 9 in 10 don't know they have it. Specific blood and urine tests are needed to check for CKD. CKD can be treated (the earlier treatment starts the better). MORE INFORMATION For Everyone For Health Care Providers For Public Health About your kidneys and CKD Your hard-working kidneys Your kidneys, each just the size of a computer mouse, filter all the blood in your body every 30 minutes. They work hard to remove wastes, toxins, and excess fluid. They also: Help control blood pressure. Signal the body to make red blood ce

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode([c["text"] for c in chunks], normalize_embeddings=True, show_progress_bar=True)
print(embeddings.shape)

with open(OUT_DIR / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)
np.save(OUT_DIR / "embeddings.npy", embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

(154, 384)


In [5]:
def top3(q):
    qv = model.encode([q], normalize_embeddings=True)[0]
    s = embeddings @ qv
    for i in s.argsort()[::-1][:3]:
        print(f"  {s[i]:.2f} | {chunks[i]['source']} | p{chunks[i]['page']} | {chunks[i]['text'][:90]}...")

for q in ["What is eGFR?", "Which tests check for kidney disease?", "What is specific gravity?"]:
    print("\nQ:", q)
    top3(q)


Q: What is eGFR?
  0.52 | KDIGO 2024 Executive Summary.pdf | p15 | Practice Point 4.2.2: For most people and clinical settings, validated eGFR equations usin...
  0.51 | KDIGO 2024 Executive Summary.pdf | p13 | Importantly, tables are provided describing “expected” values for common laboratory tests ...
  0.50 | KDIGO 2024 Executive Summary.pdf | p15 | Practice Point 4.2.4: In people with extremes of body weight, eGFR nonindexed for body sur...

Q: Which tests check for kidney disease?
  0.66 | CDC Testing.pdf | p1 | Testing for Chronic Kidney Disease KEY POINTS If you have diabetes or high blood pressure,...
  0.64 | CDC Testing.pdf | p2 | These could include monitoring your blood pressure, running imaging tests, or performing a...
  0.62 | CDC Testing.pdf | p2 | As kidney disease progresses, your GFR goes down. The results of your test can mean the fo...

Q: What is specific gravity?
  0.19 | KDIGO 2024 Executive Summary.pdf | p15 | Practice Point 4.2.4: In people with extremes of b